# EasyOCR
- It is a Python library that allows you to perform Optical Character Recognition (OCR) on images. It supports multiple languages and is easy to use. In this notebook, we will demonstrate how to use EasyOCR to detect text in an image.

The return value of the `reader.readtext(image)` function is a list of tuples, where each tuple contains three elements:
1. `bbox`: A list of four points that define the bounding box around the detected text. Each point is represented as a list of two coordinates (x, y).
2. `DetectedText`: A string that contains the text that was detected in the image.
3. `confidence`: A float value that represents the confidence level of the detected text. It ranges from 0 to 1
```
[
    [ [x1,y1], [x2,y2], [x3,y3], [x4,y4] ],   # bounding box
    "detected_text",                          # text
    confidence_score                          # confidence
]

[x1,y1] -> Top-left corner of the bounding box
[x2,y2] -> Top-right corner of the bounding box
[x3,y3] -> Bottom-right corner of the bounding box
[x4,y4] -> Bottom-left corner of the bounding box
```

## For Images

In [ ]:
# ---------------------------------------
# Loading Libraries
# ---------------------------------------
import easyocr
import cv2
import matplotlib.pyplot as plt

# ---------------------------------------
# Loading Image
# ---------------------------------------
image_path = "test-en.jpg"
image = cv2.imread(image_path)

# ---------------------------------------
# Detecting Text
# ---------------------------------------
reader = easyocr.Reader(['en'])
text = reader.readtext(image)

# ---------------------------------------
# Displaying Detected Text
# ---------------------------------------
for t in text:
    bbox, text, confidence = t
    print(f"Detected Text: {text}, Confidence: {confidence}")
    top_left = tuple(map(int, bbox[0]))
    bottom_right = tuple(map(int, bbox[2]))
    cv2.rectangle(image, top_left, bottom_right, (0, 255, 0), 5)

    # Calculate text size
    font = cv2.FONT_HERSHEY_DUPLEX
    font_scale = 1.2
    thickness = 2
    text_size, baseline = cv2.getTextSize(text, font, font_scale, thickness)
    text_width, text_height = text_size

    # Add padding
    pad_x, pad_y = 10, 10
    rect_top_left = (top_left[0], top_left[1] - text_height - pad_y)
    rect_bottom_right = (top_left[0] + text_width + pad_x, top_left[1])

    # Draw filled rectangle for text background
    cv2.rectangle(image, rect_top_left, rect_bottom_right, (0, 255, 0), -1)

    # Draw text
    text_org = (top_left[0] + 5, top_left[1] - 5)
    cv2.putText(image, text, text_org, font, font_scale, (0,0,0), thickness, cv2.LINE_AA)

# Display using matplotlib instead of cv2.imshow
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.imshow(image_rgb)
plt.axis('off')
plt.title("Text Detection")
plt.show()


## For Videos

In [ ]:
# ---------------------------------------
# Loading Libraries
# ---------------------------------------
import easyocr
import cv2
import torch

print(f"PyTorch GPU Available: {torch.cuda.is_available()}")
import cv2
print("GUI:", "YES" if "GUI: NONE" not in cv2.getBuildInformation() else "NO")

# ---------------------------------------
# Loading Video
# ---------------------------------------
cap = cv2.VideoCapture(0)
reader = easyocr.Reader(['en'], gpu=True)  # Enable GPU

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Detecting Text
    text = reader.readtext(frame)

    # Displaying Detected Text
    for t in text:
        bbox, text, confidence = t
        top_left = tuple(map(int, bbox[0]))
        bottom_right = tuple(map(int, bbox[2]))
        cv2.rectangle(frame, top_left, bottom_right, (0, 255, 0), 5)
        font = cv2.FONT_HERSHEY_DUPLEX
        font_scale = 1.2
        thickness = 2
        text_size, baseline = cv2.getTextSize(text, font, font_scale, thickness)
        text_width, text_height = text_size
        pad_x, pad_y = 10, 10
        rect_top_left = (top_left[0], top_left[1] - text_height - pad_y)
        rect_bottom_right = (top_left[0] + text_width + pad_x, top_left[1])
        cv2.rectangle(frame, rect_top_left, rect_bottom_right, (0, 255, 0), -1)
        text_org = (top_left[0] + 5, top_left[1] - 5)
        cv2.putText(frame, text, text_org, font, font_scale, (0,0,0), thickness, cv2.LINE_AA)

    cv2.imshow('Video Text Detection', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


---